In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import split_scale_data as ssd
import perform_GBFS as pgbfs
import perform_feature_analysis as pfa
import perform_feature_engineering as pfe
import perform_multicollinearity_reduction as pmr
import perform_recursive_feature_elimination as prfe
import perform_dummy_test as pdt
import perform_bayesian_optimization as pbo
import perform_final_figure as pff

import os

file_path = os.getcwd() # Path to the folder containing the data files. Default is the current directory
file_name = '2d_combined'
target = 'bandgap' # 'is_stable' or 'is_metal' or 'bandgap'
objective = 'regression' # 'regression'
unit = 'eV' # "eV" or None
boosting_method = 'lightGBM' # 'lightGBM' or 'XGBoost'

target_filter = lambda x: x > 0.2 # None # Filters the target variable. # None or lambda x: x > 0.2
dataset_filter = None #  # These entries will be excluded from the training.
features_filter = None # List of features to be filtered out of data prior to GBFS
stability_filter = False # Filters based on thermodynamic stability boolean value
metallicity_filter = False # Filters based on metallicity boolean value

In [ ]:
df_train, df_test, feature_list = ssd.split_scale_data( 
                      file_path=file_path, 
                      file_name=file_name, 
                      target=target, 
                      objective=objective,
                      target_filter=target_filter,
                      features_filter=features_filter,
                      dataset_filter=dataset_filter,
                      stability_filter=stability_filter,
                      metallicity_filter=metallicity_filter,
                      test_size=0.2,
                      path_to_expt=None
                    )

In [ ]:
pgbfs.perform_GBFS(
                    file_path=file_path, 
                    file_name=file_name, 
                    target=target, 
                    objective=objective,
                    cv_folds=10,
                    oversampled=False
                  )

In [ ]:
pfa.perform_feature_analysis( # df_pfa, df_anova, df_mi, df_chi2, df_ld = 
                              file_path=file_path, 
                              file_name=file_name, 
                              target=target, 
                              objective=objective
                            )

In [ ]:
df_train_pfe, df_test_pfe, new_cols =  pfe.perform_feature_engineering( 
                                  file_path=file_path, 
                                  file_name=file_name, 
                                  target=target, 
                                  objective=objective,
                                  no_of_top_features = 5
                                )

In [ ]:
pmr.perform_multicollinearity_reduction( # final_features = 
                                          file_path=file_path, 
                                          file_name=file_name, 
                                          target=target, 
                                          objective=objective,
                                          no_of_relevant_features = 200,
                                          correlation_threshold = 0.9,
                                          max_link_threshold = 5
                                        )

In [ ]:
prfe.perform_recursive_feature_elimination( # RFE_features = 
                                              file_path=file_path, 
                                              file_name=file_name, 
                                              target=target, 
                                              objective=objective,
                                              threshold=1, 
                                              cv_fold=10, 
                                              boosting_method=boosting_method
                                            )

In [ ]:
pdt.perform_dummy_test( # df_pred, feature_score = 
                        file_path=file_path, 
                        file_name=file_name, 
                        target=target, 
                        objective=objective,
                        unit=unit,
                        threshold=1, 
                        learning_rate=0.1, 
                        n_estimators=350, 
                        num_leaves=40, 
                        max_depth=-1, 
                        random_state=42, 
                        no_of_features_to_plot=25,
                        model_type=boosting_method
                        )

In [ ]:
pbo.perform_bayesian_optimization(
                                    file_path=file_path, 
                                    file_name=file_name, 
                                    target=target, 
                                    objective=objective, 
                                    scaled=False,
                                    boosting_method=boosting_method, 
                                    optimization_method='bayesian', 
                                    n_calls=100,
                                    strategy='weighted', 
                                    adjusted=False
                                  )

In [ ]:
# # df_pred now contains the column for 'residual_error' calculated in the perform_final_figure

final_feature_score, df_pred = pff.perform_final_figure(
                          file_path=file_path, 
                          file_name=file_name, 
                          target=target, 
                          objective=objective,
                          unit=unit, 
                          boosting_type='gbdt', 
                          importance_type='gain',
                          scaled=False,
                          no_of_features_to_plot=25,
                          max_depth=-1,
                          random_state=42
                        )